# LeetCode 232 — Implement Queue using Stacks bằng Python

- Link đề: https://leetcode.com/problems/implement-queue-using-stacks/
- Độ khó: **Easy** · Chủ đề: Stack, Queue, Design (thiết kế class)

Khung LeetCode cần viết (bài **thiết kế**: viết cả một class, không phải `class Solution`):

```python
class MyQueue:

    def __init__(self):


    def push(self, x: int) -> None:


    def pop(self) -> int:


    def peek(self) -> int:


    def empty(self) -> bool:

```

**Mục tiêu học**

1. Phân biệt **stack (LIFO)** và **queue (FIFO)**, biết thao tác hợp lệ của mỗi loại.
2. Ôn cách viết `class`, `__init__`, `self`, thuộc tính và phương thức trong Python.
3. Hiểu ý tưởng "đổ stack sang stack khác thì thứ tự bị đảo ngược".
4. Xây 2 thiết kế: chuyển mỗi lần push (`push O(n)`) và **lười chuyển** (`amortized O(1)`).
5. Hiểu khái niệm **amortized** (chi phí trung bình trên chuỗi thao tác) — câu Follow-up của đề.

## 1. Đề bài và ví dụ

Cài đặt hàng đợi FIFO **chỉ dùng hai stack**. Class `MyQueue` hỗ trợ:

- `push(x)`: thêm `x` vào **cuối** hàng đợi.
- `pop()`: lấy ra và trả phần tử ở **đầu** hàng đợi.
- `peek()`: trả phần tử ở đầu (không lấy ra).
- `empty()`: `True` nếu hàng đợi rỗng.

Ràng buộc: chỉ được dùng thao tác chuẩn của stack — **push lên đỉnh, peek/pop ở đỉnh, size, is empty**. Trong Python mô phỏng stack bằng `list` với `append`, `pop()`, `[-1]`, `len`.

Ví dụ:

```text
Input : ["MyQueue","push","push","peek","pop","empty"]
        [[],        [1],   [2],   [],    [],   []]
Output: [null,      null,  null,  1,     1,    false]
```

Constraints: `1 <= x <= 9`; tối đa 100 lệnh; mọi lệnh `pop`/`peek` đều hợp lệ (hàng đợi không rỗng khi gọi).

**Follow-up:** mỗi thao tác **amortized O(1)** — tức `n` thao tác tốn tổng `O(n)` dù có lần riêng lẻ tốn hơn.

## 2. Nền tảng: Stack và Queue

| | Stack (ngăn xếp) | Queue (hàng đợi) |
|---|---|---|
| Nguyên tắc | **LIFO** — vào sau ra trước | **FIFO** — vào trước ra trước |
| Hình ảnh | chồng đĩa | hàng người xếp mua vé |
| Thêm | đỉnh | cuối hàng |
| Lấy | đỉnh | đầu hàng |

### 2.1. Stack bằng `list`

In [1]:
stack = []
stack.append(1)     # push 1
stack.append(2)     # push 2
stack.append(3)     # push 3
print("stack:", stack, "| đỉnh (peek) =", stack[-1])
print("pop ->", stack.pop())    # lấy 3: vào sau, ra trước
print("pop ->", stack.pop())
print("còn lại:", stack, "| rỗng?", len(stack) == 0)

stack: [1, 2, 3] | đỉnh (peek) = 3
pop -> 3
pop -> 2
còn lại: [1] | rỗng? False


- `append(x)` thêm vào cuối list = đỉnh stack: `O(1)` (amortized).
- `pop()` không đối số lấy phần tử cuối: `O(1)`.
- `stack[-1]` xem đỉnh không lấy ra.
- `not stack` là `True` khi list rỗng.

### 2.2. Hành vi Queue mà ta cần đạt

Push 1, 2, 3 thì pop phải ra 1, rồi 2, rồi 3. Stack lại cho 3, 2, 1 — **ngược** hoàn toàn.

In [2]:
from collections import deque
q = deque()          # deque: cấu trúc queue chuẩn của Python (chỉ để tham chiếu)
for x in [1, 2, 3]:
    q.append(x)
print("queue pop:", q.popleft(), q.popleft(), q.popleft())

queue pop: 1 2 3


`collections.deque` là hàng đợi hai đầu, `popleft()` `O(1)`. Đề **không cho** dùng nó làm queue, nhưng ta dùng làm "đáp án chuẩn" (oracle) khi test.

### 2.3. Ôn nhanh `class` trong Python

- `class MyQueue:` định nghĩa kiểu dữ liệu mới.
- `__init__(self)` chạy khi tạo đối tượng `MyQueue()`; dùng để khởi tạo thuộc tính.
- `self.xxx` là thuộc tính riêng của từng đối tượng.
- Mỗi phương thức nhận `self` làm tham số đầu.

In [3]:
class Counter:
    def __init__(self):
        self.value = 0          # thuộc tính
    def inc(self) -> None:      # phương thức
        self.value += 1
    def get(self) -> int:
        return self.value

c1, c2 = Counter(), Counter()
c1.inc(); c1.inc(); c2.inc()
print(c1.get(), c2.get())       # 2 1 -> hai đối tượng độc lập

2 1


## 3. Cách ngây thơ — một list, lấy đầu bằng `pop(0)`

Người mới hay nghĩ: cứ `append` vào cuối, `pop(0)` lấy đầu. Đúng hành vi queue, nhưng **vi phạm đề**: `pop(0)` không phải thao tác stack, và nó tốn `O(n)` vì phải dời mọi phần tử còn lại sang trái.

In [4]:
class NaiveQueue:
    def __init__(self):
        self.data = []
    def push(self, x): self.data.append(x)
    def pop(self): return self.data.pop(0)     # O(n), không phải thao tác stack
    def peek(self): return self.data[0]        # đọc đáy stack, không hợp lệ
    def empty(self): return not self.data

nq = NaiveQueue()
for x in [1, 2, 3]:
    nq.push(x)
print(nq.peek(), nq.pop(), nq.pop(), nq.empty())

1 1 2 False


In [5]:
import timeit
setup = "data = list(range(200000))"
t_pop0 = timeit.timeit("data.pop(0); data.append(0)", setup=setup, number=2000)
t_pop = timeit.timeit("data.pop(); data.append(0)", setup=setup, number=2000)
print(f"pop(0) chậm hơn pop() khoảng {t_pop0 / t_pop:.0f} lần với list 200000 phần tử")

pop(0) chậm hơn pop() khoảng 1082 lần với list 200000 phần tử


(Con số cụ thể tùy máy, nhưng `pop(0)` luôn chậm hơn rõ rệt.) Ta cần cách chỉ dùng thao tác ở **đỉnh**.

## 4. Insight — đổ stack này sang stack kia thì thứ tự bị đảo

Nếu pop hết stack A và push từng phần tử sang stack B, phần tử ở **đáy** A (vào sớm nhất) sẽ thành **đỉnh** B.

```text
A = [1, 2, 3]  (đỉnh là 3)
đổ: pop 3 -> B=[3]; pop 2 -> B=[3,2]; pop 1 -> B=[3,2,1]
B = [3, 2, 1]  (đỉnh là 1)  -> pop B cho 1, 2, 3: đúng thứ tự FIFO!
```

Đảo ngược hai lần = đúng thứ tự. Đây là toàn bộ bí mật của bài.

In [6]:
A = [1, 2, 3]
B = []
while A:
    B.append(A.pop())
    print("A =", A, "| B =", B)
print("Pop từ B:", [B.pop() for _ in range(3)])

A = [1, 2] | B = [3]
A = [1] | B = [3, 2]
A = [] | B = [3, 2, 1]
Pop từ B: [1, 2, 3]


`[B.pop() for _ in range(3)]` là **list comprehension**: gọi `B.pop()` 3 lần, gom kết quả thành list. `_` là tên biến quy ước khi ta không dùng tới giá trị đếm.

## 5. Cách 2 — Luôn giữ stack ở "thứ tự queue" (push tốn O(n))

Ý tưởng: giữ `s1` sao cho **đỉnh của `s1` luôn là đầu hàng đợi**. Khi push `x`:

1. Đổ hết `s1` sang `s2`.
2. Push `x` vào `s1` (giờ `x` nằm đáy).
3. Đổ hết `s2` về `s1`.

Pop/peek chỉ cần thao tác đỉnh `s1`: `O(1)`.

In [7]:
class MyQueuePushHeavy:
    def __init__(self):
        self.s1 = []   # đỉnh s1 = đầu hàng đợi
        self.s2 = []   # stack phụ

    def push(self, x: int) -> None:
        while self.s1:
            self.s2.append(self.s1.pop())
        self.s1.append(x)
        while self.s2:
            self.s1.append(self.s2.pop())

    def pop(self) -> int:
        return self.s1.pop()

    def peek(self) -> int:
        return self.s1[-1]

    def empty(self) -> bool:
        return not self.s1

q = MyQueuePushHeavy()
for x in [1, 2, 3]:
    q.push(x)
    print("sau push", x, "-> s1 =", q.s1)
print(q.peek(), q.pop(), q.pop(), q.empty(), q.pop(), q.empty())

sau push 1 -> s1 = [1]
sau push 2 -> s1 = [2, 1]
sau push 3 -> s1 = [3, 2, 1]
1 1 2 False 3 True


### Nút thắt của cách 2

Mỗi `push` đổ toàn bộ `s1` qua lại hai lần → `O(n)` mỗi push, `n` lần push tốn `O(n²)`. Ta đang đổ dữ liệu **quá sớm và quá thường xuyên**: phần lớn các lần đổ là thừa vì chưa ai cần đọc đầu hàng.

In [8]:
def moves_push_heavy(n):
    q = MyQueuePushHeavy()
    moves = 0
    for x in range(n):
        moves += 2 * len(q.s1)   # đổ sang rồi đổ về
        q.push(x)
    return moves

for n in [10, 100, 1000]:
    print(f"n={n:>4}: số lần di chuyển phần tử = {moves_push_heavy(n)}")

n=  10: số lần di chuyển phần tử = 90
n= 100: số lần di chuyển phần tử = 9900
n=1000: số lần di chuyển phần tử = 999000


Số lần di chuyển tăng theo `n²` (`n(n-1)`).

## 6. Insight chính — chỉ đổ khi `out` rỗng (lười)

Dùng hai stack với vai trò cố định:

- `inbox` (`in_stack`): nhận mọi `push`.
- `outbox` (`out_stack`): phục vụ `pop`/`peek`; đỉnh của nó là đầu hàng.

Quy tắc vàng: **chỉ khi `outbox` rỗng** mới đổ toàn bộ `inbox` sang `outbox`. Nếu `outbox` còn phần tử, đỉnh của nó chắc chắn là phần tử cũ nhất, không cần làm gì.

Tại sao không được đổ khi `outbox` còn phần tử? Vì phần tử mới từ `inbox` sẽ đè lên đỉnh `outbox`, chen ngang trước những phần tử cũ hơn → sai thứ tự.

In [9]:
inbox, outbox = [1, 2], [ ]
# đổ khi outbox rỗng -> đúng
while inbox:
    outbox.append(inbox.pop())
print("outbox =", outbox, "đỉnh =", outbox[-1])
inbox.append(3)
# nếu đổ ngay khi outbox chưa rỗng (SAI):
bad = outbox.copy()
tmp = inbox.copy()
while tmp:
    bad.append(tmp.pop())
print("đổ sai -> đỉnh =", bad[-1], "(phải là 1, nhưng ra 3)")

outbox = [2, 1] đỉnh = 1
đổ sai -> đỉnh = 3 (phải là 1, nhưng ra 3)


## 7. Lời giải chính — hai stack lười chuyển

Ghép từng mảnh:

- `push(x)`: `inbox.append(x)` — luôn `O(1)`.
- Hàm phụ `_move()`: nếu `outbox` rỗng thì đổ hết `inbox` sang.
- `peek()`: `_move()` rồi trả `outbox[-1]`.
- `pop()`: `_move()` rồi `outbox.pop()`.
- `empty()`: rỗng khi **cả hai** stack rỗng.

Tên bắt đầu bằng `_` (như `_move`) là quy ước "phương thức nội bộ", người dùng class không nên gọi trực tiếp.

In [10]:
class MyQueueAmortized:
    def __init__(self):
        self.inbox = []
        self.outbox = []

    def _move(self) -> None:
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def push(self, x: int) -> None:
        self.inbox.append(x)

    def pop(self) -> int:
        self._move()
        return self.outbox.pop()

    def peek(self) -> int:
        self._move()
        return self.outbox[-1]

    def empty(self) -> bool:
        return not self.inbox and not self.outbox

q = MyQueueAmortized()
q.push(1); q.push(2)
print(q.peek(), q.pop(), q.empty())   # 1 1 False

1 1 False


## 8. Trace từng bước

Chuỗi lệnh: `push 1, push 2, peek, push 3, pop, pop, push 4, pop, pop, empty`.

| Lệnh | Có đổ? | inbox sau | outbox sau (đỉnh ở cuối) | Trả về |
|---|---|---|---|---|
| push 1 | không | [1] | [] | – |
| push 2 | không | [1,2] | [] | – |
| peek | **có** (outbox rỗng) | [] | [2,1] | 1 |
| push 3 | không | [3] | [2,1] | – |
| pop | không (outbox còn) | [3] | [2] | 1 |
| pop | không | [3] | [] | 2 |
| push 4 | không | [3,4] | [] | – |
| pop | **có** | [] | [4] | 3 |
| pop | không | [] | [] | 4 |
| empty | – | [] | [] | True |

Chú ý dòng `push 3`: 3 nằm chờ ở `inbox` trong khi 1, 2 vẫn được phục vụ trước từ `outbox`.

In [11]:
def trace_queue(ops, args):
    q = None
    out = []
    for op, arg in zip(ops, args):
        if op == "MyQueue":
            q = MyQueueAmortized()
            out.append(None)
            print(f"{'MyQueue()':<10} inbox={q.inbox} outbox={q.outbox}")
            continue
        before_out = len(q.outbox)
        res = getattr(q, op)(*arg)
        moved = op in ("pop", "peek") and before_out == 0
        out.append(res)
        label = f"{op}({arg[0]})" if arg else f"{op}()"
        print(f"{label:<10} {'[ĐỔ] ' if moved else '      '}inbox={q.inbox} outbox={q.outbox} -> {res}")
    return out

ops = ["MyQueue", "push", "push", "peek", "push", "pop", "pop", "push", "pop", "pop", "empty"]
args = [[], [1], [2], [], [3], [], [], [4], [], [], []]
print(trace_queue(ops, args))

MyQueue()  inbox=[] outbox=[]
push(1)          inbox=[1] outbox=[] -> None
push(2)          inbox=[1, 2] outbox=[] -> None
peek()     [ĐỔ] inbox=[] outbox=[2, 1] -> 1
push(3)          inbox=[3] outbox=[2, 1] -> None
pop()            inbox=[3] outbox=[2] -> 1
pop()            inbox=[3] outbox=[] -> 2
push(4)          inbox=[3, 4] outbox=[] -> None
pop()      [ĐỔ] inbox=[] outbox=[4] -> 3
pop()            inbox=[] outbox=[] -> 4
empty()          inbox=[] outbox=[] -> True
[None, None, None, 1, None, 1, 2, None, 3, 4, True]


Hai cú pháp mới trong `trace_queue`:

- `zip(ops, args)` ghép từng cặp phần tử cùng vị trí của hai list.
- `getattr(q, "push")` lấy phương thức theo **tên chuỗi**; `getattr(q, op)(*arg)` gọi nó với các đối số trong list `arg` (`*arg` "trải" list thành đối số). Đây cũng là cách LeetCode chạy test của bài thiết kế.

In [12]:
print(getattr([], "__len__")(), "|", max(*[3, 9, 4]))   # *list trải phần tử thành đối số

0 | 9


## 9. Code nộp LeetCode

In [13]:
class MyQueue:

    def __init__(self):
        self.in_stack = []
        self.out_stack = []

    def push(self, x: int) -> None:
        self.in_stack.append(x)

    def pop(self) -> int:
        self.peek()                      # đảm bảo out_stack có đầu hàng
        return self.out_stack.pop()

    def peek(self) -> int:
        if not self.out_stack:
            while self.in_stack:
                self.out_stack.append(self.in_stack.pop())
        return self.out_stack[-1]

    def empty(self) -> bool:
        return not self.in_stack and not self.out_stack


# Your MyQueue object will be instantiated and called as such:
obj = MyQueue()
obj.push(1)
obj.push(2)
print(obj.peek(), obj.pop(), obj.empty())

1 1 False


Mẹo nhỏ: `pop()` gọi lại `peek()` để tái dùng logic đổ stack, tránh viết trùng.

## 10. Vì sao đúng? (invariant)

Gọi hàng đợi logic là dãy các phần tử theo thứ tự push. **Invariant:**

> Hàng đợi = (`out_stack` đọc từ **đỉnh xuống đáy**) nối với (`in_stack` đọc từ **đáy lên đỉnh**).

- `push(x)`: thêm vào đỉnh `in_stack` = cuối của phần sau → đúng là cuối hàng.
- Đổ khi `out_stack` rỗng: `in_stack` (đáy→đỉnh) đảo thành `out_stack` (đỉnh→đáy), giữ nguyên dãy logic.
- `pop`/`peek` lấy đỉnh `out_stack` = phần tử đầu dãy logic = đầu hàng.
- Nếu `out_stack` không rỗng, mọi phần tử trong nó **cũ hơn** mọi phần tử trong `in_stack`, nên không cần đổ.

In [14]:
def logical_queue(q: MyQueue) -> list[int]:
    return q.out_stack[::-1] + q.in_stack

q = MyQueue()
ref = deque()
import random
rng = random.Random(232)
for step in range(30):
    if ref and rng.random() < 0.4:
        assert q.pop() == ref.popleft()
    else:
        x = rng.randint(1, 9)
        q.push(x); ref.append(x)
    assert logical_queue(q) == list(ref)
print("Invariant được giữ qua 30 thao tác ngẫu nhiên. Trạng thái cuối:", logical_queue(q))

Invariant được giữ qua 30 thao tác ngẫu nhiên. Trạng thái cuối: []


## 11. Độ phức tạp — hiểu "amortized O(1)"

Gọi `n` là số thao tác.

| Thao tác | Cách 2 (push nặng) | Cách 3 (lười) — trường hợp xấu | Cách 3 — amortized |
|---|---|---|---|
| push | `O(n)` | `O(1)` | `O(1)` |
| pop | `O(1)` | `O(n)` (khi phải đổ) | `O(1)` |
| peek | `O(1)` | `O(n)` (khi phải đổ) | `O(1)` |
| empty | `O(1)` | `O(1)` | `O(1)` |
| Space | `O(n)` | `O(n)` | `O(n)` |

**Lập luận amortized (đếm theo từng phần tử):** mỗi phần tử trong đời nó chỉ trải qua tối đa 4 thao tác stack: push vào `in`, pop khỏi `in`, push vào `out`, pop khỏi `out`. Không phần tử nào quay lại `in_stack`. Vậy `n` thao tác queue → tối đa `4n` thao tác stack → trung bình `O(1)` mỗi thao tác.

In [15]:
class CountingQueue(MyQueue):
    """Kế thừa MyQueue, đếm số thao tác stack cơ bản."""
    def __init__(self):
        super().__init__()
        self.ops = 0
    def push(self, x):
        self.ops += 1
        super().push(x)
    def peek(self):
        if not self.out_stack:
            self.ops += 2 * len(self.in_stack)   # mỗi phần tử: pop khỏi in + push vào out
        return super().peek()
    def pop(self):
        v = self.peek()
        self.ops += 1
        self.out_stack.pop()
        return v

for n in [10, 100, 1000, 10000]:
    cq = CountingQueue()
    for i in range(n):
        cq.push(i)
        if i % 3 == 2:
            cq.pop()
    while not cq.empty():
        cq.pop()
    print(f"n push = {n:>5}: tổng thao tác stack = {cq.ops:>6}  (~{cq.ops / n:.2f} mỗi push, luôn ≤ 4)")

n push =    10: tổng thao tác stack =     40  (~4.00 mỗi push, luôn ≤ 4)
n push =   100: tổng thao tác stack =    400  (~4.00 mỗi push, luôn ≤ 4)
n push =  1000: tổng thao tác stack =   4000  (~4.00 mỗi push, luôn ≤ 4)
n push = 10000: tổng thao tác stack =  40000  (~4.00 mỗi push, luôn ≤ 4)


`class CountingQueue(MyQueue)` là **kế thừa**: lớp con có sẵn mọi phương thức của lớp cha; `super().push(x)` gọi bản của lớp cha. So với cách 2 (tăng theo `n²`), cách lười tăng **tuyến tính**.

## 12. Bộ test

In [16]:
def run_design(ops, args, cls):
    obj, out = None, []
    for op, arg in zip(ops, args):
        if op == "MyQueue":
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

check("ví dụ đề",
      run_design(["MyQueue", "push", "push", "peek", "pop", "empty"], [[], [1], [2], [], [], []], MyQueue),
      [None, None, None, 1, 1, False])
check("một phần tử",
      run_design(["MyQueue", "push", "empty", "pop", "empty"], [[], [5], [], [], []], MyQueue),
      [None, None, False, 5, True])
check("xen kẽ push/pop",
      run_design(["MyQueue", "push", "push", "pop", "push", "pop", "pop"], [[], [1], [2], [], [3], [], []], MyQueue),
      [None, None, None, 1, None, 2, 3])
check("peek nhiều lần không đổi trạng thái",
      run_design(["MyQueue", "push", "peek", "peek", "pop", "empty"], [[], [9], [], [], [], []], MyQueue),
      [None, None, 9, 9, 9, True])

PASS  ví dụ đề: [None, None, None, 1, 1, False]
PASS  một phần tử: [None, None, False, 5, True]
PASS  xen kẽ push/pop: [None, None, None, 1, None, 2, 3]
PASS  peek nhiều lần không đổi trạng thái: [None, None, 9, 9, 9, True]


In [17]:
class DequeQueue:
    """Oracle dùng deque — chỉ để so sánh."""
    def __init__(self): self.d = deque()
    def push(self, x): self.d.append(x)
    def pop(self): return self.d.popleft()
    def peek(self): return self.d[0]
    def empty(self): return not self.d

for trial in range(300):
    ops, args, size = ["MyQueue"], [[]], 0
    for _ in range(rng.randint(1, 100)):
        choice = rng.choice(["push", "push", "pop", "peek", "empty"])
        if choice in ("pop", "peek") and size == 0:
            choice = "push"                   # đề đảm bảo pop/peek hợp lệ
        if choice == "push":
            ops.append("push"); args.append([rng.randint(1, 9)]); size += 1
        else:
            ops.append(choice); args.append([])
            size -= choice == "pop"
    expected = run_design(ops, args, DequeQueue)
    for cls in (MyQueue, MyQueueAmortized, MyQueuePushHeavy):
        assert run_design(ops, args, cls) == expected, (cls.__name__, ops, args)
print("300 chuỗi lệnh ngẫu nhiên: 3 cài đặt khớp oracle deque — PASS")

300 chuỗi lệnh ngẫu nhiên: 3 cài đặt khớp oracle deque — PASS


`size -= choice == "pop"`: biểu thức `choice == "pop"` là `True`/`False`, mà trong Python `True == 1`, `False == 0`, nên dòng này chỉ trừ 1 khi lệnh là `pop`.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Đổ `in` sang `out` mỗi lần pop, kể cả khi `out` chưa rỗng

In [18]:
class WrongAlwaysMove(MyQueue):
    def peek(self):
        while self.in_stack:                     # thiếu điều kiện "out rỗng"
            self.out_stack.append(self.in_stack.pop())
        return self.out_stack[-1]

print("sai :", run_design(["MyQueue", "push", "push", "pop", "push", "pop"], [[], [1], [2], [], [3], []], WrongAlwaysMove))
print("đúng:", run_design(["MyQueue", "push", "push", "pop", "push", "pop"], [[], [1], [2], [], [3], []], MyQueue))

sai : [None, None, None, 1, None, 3]
đúng: [None, None, None, 1, None, 2]


**Vì sao sai:** sau `pop` đầu, `out = [2]`; push 3 vào `in`; lần pop sau đổ 3 lên trên 2 → trả 3 thay vì 2. **Sửa:** chỉ đổ khi `out_stack` rỗng.

### Lỗi 2: `empty()` chỉ kiểm tra một stack

In [19]:
class WrongEmpty(MyQueue):
    def empty(self):
        return not self.out_stack                # quên in_stack

w = WrongEmpty(); w.push(1)
print("WrongEmpty sau push(1): empty() =", w.empty(), "(phải False)")

WrongEmpty sau push(1): empty() = True (phải False)


**Sửa:** `return not self.in_stack and not self.out_stack`.

### Lỗi 3: Dùng thuộc tính cấp class thay vì `self`

```python
class BadQueue:
    in_stack = []      # list này DÙNG CHUNG cho mọi đối tượng!
```

LeetCode tạo nhiều đối tượng trong các test; dữ liệu test trước sẽ "rò" sang test sau.

In [20]:
class BadQueue:
    in_stack = []
    def push(self, x): self.in_stack.append(x)

a, b = BadQueue(), BadQueue()
a.push(1)
print("b.in_stack =", b.in_stack, "<- b chưa push gì mà đã có dữ liệu!")

b.in_stack = [1] <- b chưa push gì mà đã có dữ liệu!


**Sửa:** khởi tạo trong `__init__` bằng `self.in_stack = []`.

### Lỗi 4: Dùng `pop(0)` hoặc `deque.popleft()`
Chạy đúng nhưng vi phạm yêu cầu "chỉ dùng thao tác stack" — phỏng vấn sẽ bị đánh trượt dù LeetCode chấm Accepted.

## 14. Mẹo nhận dạng pattern & biến thể

- "Cài cấu trúc A bằng cấu trúc B" → tìm phép biến đổi giữ thứ tự (ở đây: đảo hai lần).
- "Amortized O(1)" → thường là **làm việc nặng một cách lười**, mỗi phần tử chỉ bị xử lý hằng số lần.
- Hai stack `in/out` còn dùng cho **queue có min/max** (mỗi stack lưu kèm min), và mô hình "sliding window".

Biến thể: **Implement Stack using Queues (LeetCode 225)** — ngược lại, dùng queue làm stack: push `x` rồi xoay `len-1` phần tử phía trước ra sau.

In [21]:
class MyStackUsingQueue:
    def __init__(self):
        self.q = deque()          # chỉ dùng append (cuối) và popleft (đầu)
    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())   # xoay để x lên đầu
    def pop(self): return self.q.popleft()
    def top(self): return self.q[0]
    def empty(self): return not self.q

st = MyStackUsingQueue()
for x in [1, 2, 3]:
    st.push(x)
print(st.top(), st.pop(), st.pop(), st.pop(), st.empty())

3 3 2 1 True


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Trace bằng tay
Với chuỗi `push 5, push 6, pop, push 7, peek, pop, pop` hãy ghi `in_stack`, `out_stack` sau mỗi lệnh và giá trị trả về.

### Bài 2 — Thêm `size()`

In [22]:
class ExerciseQueueSize(MyQueue):
    def size(self) -> int:
        # TODO: số phần tử của hàng đợi
        pass

# q = ExerciseQueueSize(); q.push(1); q.push(2); q.pop()
# assert q.size() == 1

### Bài 3 — Tự viết lại MyQueue lười

In [23]:
class ExerciseMyQueue:
    def __init__(self):
        # TODO: hai list
        pass
    def push(self, x: int) -> None:
        # TODO
        pass
    def pop(self) -> int:
        # TODO: chỉ đổ khi out rỗng
        pass
    def peek(self) -> int:
        # TODO
        pass
    def empty(self) -> bool:
        # TODO
        pass

# assert run_design(["MyQueue","push","push","peek","pop","empty"], [[],[1],[2],[],[],[]], ExerciseMyQueue) == [None,None,None,1,1,False]

### Bài 4 — Queue hỗ trợ `get_min()` amortized O(1)
Mỗi stack lưu cặp `(giá_trị, min_đến_giờ)`. `get_min()` = min của hai đỉnh.

In [24]:
class ExerciseMinQueue:
    def __init__(self):
        # TODO
        pass
    def push(self, x: int) -> None:
        # TODO: in_stack lưu (x, min(x, min đỉnh in))
        pass
    def pop(self) -> int:
        # TODO: khi đổ sang out, tính lại min theo out
        pass
    def get_min(self) -> int:
        # TODO
        pass

# q = ExerciseMinQueue(); [q.push(v) for v in [3, 1, 4]]
# assert q.get_min() == 1; q.pop(); assert q.get_min() == 1; q.pop(); assert q.get_min() == 4

### Bài 5 — Đảo ngược k phần tử đầu của hàng đợi
Cho list `items` mô phỏng hàng đợi (đầu ở index 0) và `k`. Dùng **một stack** để đảo `k` phần tử đầu. Ví dụ `[1,2,3,4,5], k=3` → `[3,2,1,4,5]`.

In [25]:
def exercise_reverse_first_k(items: list[int], k: int) -> list[int]:
    # TODO: lấy k phần tử đầu vào stack, pop ra lần lượt, nối phần còn lại
    pass

# assert exercise_reverse_first_k([1, 2, 3, 4, 5], 3) == [3, 2, 1, 4, 5]

### Bài 6 — Mini challenge
Viết hàm `count_moves(ops)` nhận chuỗi lệnh (`"push"`/`"pop"`) và trả về **tổng số phần tử bị đổ** từ `in` sang `out` trong cách lười. Kiểm tra rằng kết quả ≤ số lệnh push.

In [26]:
def exercise_count_moves(ops: list[str]) -> int:
    # TODO: mô phỏng chỉ bằng hai biến đếm kích thước in/out
    pass

# assert exercise_count_moves(["push", "push", "pop", "push", "pop", "pop"]) == 3

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Áp quy tắc "đổ khi out rỗng" | pop đầu tiên đổ [5,6] | out=[6,5] → pop 5 |
| 2 | Tổng hai stack | `len()` | `len(self.in_stack) + len(self.out_stack)` |
| 3 | Chép khung mục 9 | `peek` lo việc đổ | `pop` gọi `peek` |
| 4 | Min của queue = min của 2 phần | mỗi phần tử stack là tuple | khi đổ: `cur = v if not out else min(v, out[-1][1])` |
| 5 | Stack đảo thứ tự | `stack = items[:k]` | `stack[::-1] + items[k:]` hoặc pop lặp |
| 6 | Chỉ cần kích thước | đổ khi out=0 → `moves += in; out += in; in = 0` | pop: `out -= 1` |

## 17. Đáp án tham khảo

Bài 1: `push5` in=[5]; `push6` in=[5,6]; `pop` đổ → out=[6,5], pop → trả 5, out=[6]; `push7` in=[7]; `peek` → 6; `pop` → 6, out=[]; `pop` đổ → out=[7], trả 7.

In [27]:
class AnswerQueueSize(MyQueue):
    def size(self):
        return len(self.in_stack) + len(self.out_stack)

class AnswerMyQueue:
    def __init__(self):
        self.a, self.b = [], []
    def push(self, x):
        self.a.append(x)
    def peek(self):
        if not self.b:
            while self.a:
                self.b.append(self.a.pop())
        return self.b[-1]
    def pop(self):
        self.peek()
        return self.b.pop()
    def empty(self):
        return not self.a and not self.b

class AnswerMinQueue:
    def __init__(self):
        self.inn, self.out = [], []          # phần tử là (value, min_so_far)
    def push(self, x):
        m = x if not self.inn else min(x, self.inn[-1][1])
        self.inn.append((x, m))
    def _move(self):
        if not self.out:
            while self.inn:
                v, _ = self.inn.pop()
                m = v if not self.out else min(v, self.out[-1][1])
                self.out.append((v, m))
    def pop(self):
        self._move()
        return self.out.pop()[0]
    def get_min(self):
        cands = [s[-1][1] for s in (self.inn, self.out) if s]
        return min(cands)

def answer_reverse_first_k(items, k):
    stack = []
    for x in items[:k]:
        stack.append(x)
    head = []
    while stack:
        head.append(stack.pop())
    return head + items[k:]

def answer_count_moves(ops):
    size_in = size_out = moves = 0
    for op in ops:
        if op == "push":
            size_in += 1
        else:
            if size_out == 0:
                moves += size_in
                size_out, size_in = size_in, 0
            size_out -= 1
    return moves

q = AnswerQueueSize(); q.push(1); q.push(2); q.pop()
assert q.size() == 1
assert run_design(["MyQueue", "push", "push", "peek", "pop", "empty"], [[], [1], [2], [], [], []], AnswerMyQueue) == [None, None, None, 1, 1, False]
assert run_design(["MyQueue", "push", "push", "pop", "push", "peek", "pop", "pop"], [[], [5], [6], [], [7], [], [], []], AnswerMyQueue) == [None, None, None, 5, None, 6, 6, 7]
mq = AnswerMinQueue()
for v in [3, 1, 4]:
    mq.push(v)
assert mq.get_min() == 1; mq.pop(); assert mq.get_min() == 1; mq.pop(); assert mq.get_min() == 4
for _ in range(200):          # đối chiếu min-queue với deque
    mq, ref = AnswerMinQueue(), deque()
    for _ in range(40):
        if ref and rng.random() < 0.4:
            assert mq.pop() == ref.popleft()
        else:
            v = rng.randint(-9, 9); mq.push(v); ref.append(v)
        if ref:
            assert mq.get_min() == min(ref)
assert answer_reverse_first_k([1, 2, 3, 4, 5], 3) == [3, 2, 1, 4, 5]
assert answer_count_moves(["push", "push", "pop", "push", "pop", "pop"]) == 3
print("Đáp án tham khảo: tất cả test PASS")

Đáp án tham khảo: tất cả test PASS


## 18. Template nhớ nhanh + checklist submit

```python
class MyQueue:
    def __init__(self):
        self.in_stack, self.out_stack = [], []
    def push(self, x):
        self.in_stack.append(x)
    def peek(self):
        if not self.out_stack:                 # CHỈ đổ khi out rỗng
            while self.in_stack:
                self.out_stack.append(self.in_stack.pop())
        return self.out_stack[-1]
    def pop(self):
        self.peek()
        return self.out_stack.pop()
    def empty(self):
        return not self.in_stack and not self.out_stack
```

**Checklist**

- [ ] Tên class đúng `MyQueue`, đủ 4 phương thức `push/pop/peek/empty`.
- [ ] Thuộc tính khởi tạo trong `__init__` bằng `self.` (không dùng biến cấp class).
- [ ] Chỉ dùng `append`, `pop()`, `[-1]`, `len`/`not` trên list (không `pop(0)`, không `deque`).
- [ ] Chỉ đổ `in → out` khi `out` rỗng.
- [ ] `empty()` kiểm tra **cả hai** stack.
- [ ] Giải thích được amortized O(1): mỗi phần tử bị di chuyển tối đa một lần.